#  Causal Primary Econometric Analysis

This notebook is for a causal analysis. It includes explicit ambition, identification, specification, results reporting, interpretation, and threats to validity.

## 1. Analysis Ambition

This notebook reports the causal ambition of the primary econometric analysis. The research question asks whether increases in female secondary school enrolment affect fertility rates in a panel of Sub-Saharan African countries.

The analysis is explicitly causal: it attempts to estimate the within-country effect of female secondary enrolment on fertility while controlling for country-specific fixed factors and common year shocks. The goal is not only to describe an association, but to assess whether changes in enrolment are linked to changes in fertility rates after accounting for unobserved heterogeneity across countries and years.

The primary estimated effect is the coefficient on `Female_Secondary_Enrollment_Rate` in a two-way fixed effects regression.

## 2. Data and Sample

- Outcome variable: `Fertility_Rate` (total fertility rate, births per woman).
- Main exposure variable: `Female_Secondary_Enrollment_Rate` (female secondary school enrolment rate, percent of eligible population).
- Panel controls: country fixed effects and year fixed effects are used instead of additional observed covariates.
- Sample: country-year observations from Malawi, Rwanda, Burkina Faso, and Mali.
- Data source: cleaned World Bank indicator panel in `../Data.clean/panel_fixed_effects_data.csv`.

The analysis sample includes 70 country-year observations after dropping missing values for the two key variables. The panel covers years 2001 through 2023 for the included countries, with a balanced regional focus on four Sub-Saharan African economies.

## 3. Identification Strategy and Assumption

The identifying assumption is that, after controlling for country fixed effects and year fixed effects, remaining variation in `Female_Secondary_Enrollment_Rate` is as good as random with respect to `Fertility_Rate`.

This requires:
- no omitted time-varying confounder that affects both changes in female secondary enrolment and changes in fertility within countries,
- no reverse causality where fertility changes directly drive female secondary enrolment within the sample window,
- and stable measurement of the key indicators across countries and years.

The strategy relies on two-way fixed effects because it removes unobserved time-invariant differences across countries and common shocks in each year. The causal interpretation therefore rests on the assumption that any remaining bias is driven by time-varying factors that are not captured by the fixed effects.


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

panel = pd.read_csv('../Data.clean/panel_fixed_effects_data.csv')
panel = panel.dropna(subset=['Female_Secondary_Enrollment_Rate', 'Fertility_Rate'])

print('Number of observations:', len(panel))
print('Country counts:')
print(panel['Country_Name'].value_counts())
print('Year range:', panel['Year'].min(), 'to', panel['Year'].max())
print('\nVariable summary:')
print(panel[['Female_Secondary_Enrollment_Rate', 'Fertility_Rate']].describe())

formula = 'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)'
model = smf.ols(formula, data=panel).fit(
    cov_type='cluster',
    cov_kwds={'groups': panel['Country_Name']}
)

print('\nRegression summary:')
print(model.summary())

coef = model.params['Female_Secondary_Enrollment_Rate']
se = model.bse['Female_Secondary_Enrollment_Rate']
pval = model.pvalues['Female_Secondary_Enrollment_Rate']
print(f"\nEstimated effect: {coef:.4f} (SE = {se:.4f}, p = {pval:.3f})")

: 

## 4. Econometric Specification 

**Functional form:** Linear OLS regression with two-way fixed effects.

**Regression equation:**
`Fertility_Rate_it = alpha_i + gamma_t + beta * Female_Secondary_Enrollment_Rate_it + u_it`

where:
- `Fertility_Rate_it` is the total fertility rate for country `i` in year `t`,
- `alpha_i` is a country fixed effect capturing time-invariant country-level differences (e.g., geography, cultural factors),
- `gamma_t` is a year fixed effect capturing common shocks affecting all countries (e.g., global economic trends),
- `Female_Secondary_Enrollment_Rate_it` is the female secondary school enrollment rate, measured as a percent,
- `beta` is the parameter of interest (the within-country, within-year effect of enrollment on fertility),
- `u_it` is the error term.

**Regressors:**
- Main treatment: `Female_Secondary_Enrollment_Rate` (continuous, percent).
- Fixed effects: `C(Country_Name)` with 4 levels (Malawi, Rwanda, Burkina Faso, Mali) and `C(Year)` with 23 levels (2001–2023).
- No additional observed covariates are included; instead, fixed effects absorb time-invariant heterogeneity and common trends.

**Justification for regressors:** Country fixed effects remove unobserved time-invariant differences across countries that may affect both enrollment and fertility (e.g., institutional quality, religious composition). Year fixed effects remove global shocks and secular fertility trends that affect all countries simultaneously, improving causal identification within countries.

**Sample:**
- Panel of 4 Sub-Saharan African countries: Malawi, Rwanda, Burkina Faso, Mali.
- Years 2001–2023 (23 years).
- Total observations: 70 (after dropping missing values for the two key variables).
- Justification: Countries selected based on data availability and regional representation in Sub-Saharan Africa.

**Error structure:**
- Standard errors are clustered by country to allow arbitrary correlation of errors within each country over time.
- Justification: Panel data often exhibits serial correlation within units; clustering corrects standard errors for this dependence and yields valid inference on the treatment coefficient.


## 5. Regression Table

| Specification | Female Secondary Enrollment coefficient | Clustered Std. Error | p-value | N | R^2 |
|---|---|---|---|---|---|
| Country + year fixed effects | -0.0015 | 0.0231 | 0.948 | 70 | 0.951 |
| Country fixed effects only | -0.0562 | 0.0076 | <0.001 | 70 | 0.867 |
| Year fixed effects only | -0.0572 | 0.0185 | 0.002 | 70 | 0.594 |
| Malawi-only OLS | -0.1332 | 0.0177 | <0.001 | 19 | 0.720 |

This regression table reports the key coefficient on female secondary enrolment across four specifications. Standard errors are clustered by country for the panel regressions and are reported in the separate column. The main two-way fixed effects model is the preferred specification because it removes both time-invariant country heterogeneity and common year shocks.

Key points from the table:
- The main country + year fixed effects estimate is essentially zero and statistically insignificant. The coefficient of -0.0015 has a large standard error relative to its magnitude, producing a p-value of 0.948. This suggests no reliable within-country association between female secondary enrolment and fertility changes once country and year effects are controlled.
- The country fixed effects only model shows a stronger negative association (-0.0562) with a highly significant p-value. The change in magnitude and significance compared to the main model indicates that common year shocks matter: omitting year effects lets secular trends drive the result.
- The year fixed effects only model also yields a negative and statistically significant coefficient (-0.0572), but it ignores persistent country-level differences. This again highlights that the choice of fixed effects materially affects the estimated relationship.
- The Malawi-only OLS result is the most negative (-0.1332) and statistically significant, but it is based on only 19 observations and therefore is less generalizable than the broader panel specifications.

The R^2 values show that the two-way fixed effects model explains most of the variation in fertility rates (0.951) by absorbing country and year-level variation. The lower R^2 in the year-only model indicates that country-specific intercepts capture a large portion of the systematic differences in fertility across countries.

## 6. Interpretation

The point estimate for `Female_Secondary_Enrollment_Rate` is negative and very close to zero. A one percentage point increase in female secondary enrolment is associated with a `-0.0015` change in fertility rate, holding fixed differences between countries and common year shocks constant.

This effect is economically negligible and statistically insignificant (`p = 0.948`). The data do not provide evidence that within-country changes in female secondary enrolment systematically change fertility rates over the sample period when the model conditions are satisfied.

The result should be read as a conditional association in the fixed effects framework; it is not evidence of a large causal fertility response to enrolment changes in this specific four-country panel.

## 7. Threats to Validity

Key threats for the causal interpretation include:
- time-varying omitted variables such as economic growth, public health investments, or fertility-related policies that co-move with female school enrolment,
- measurement error in World Bank indicator series for enrolment or fertility,
- a small sample with only four countries and 70 observations, which limits statistical power and increases sensitivity to outliers,
- potential multicollinearity from the country and year fixed effects when the panel is short.
- Data Limitatons stemming from incomplete datasets for some countries due to missing variables during unstable periods in the country

What this analysis does to reduce threats:
- uses country fixed effects to absorb unobserved, time-invariant country characteristics,
- uses year fixed effects to absorb common secular shocks across all countries,
- clusters standard errors by country to allow within-country error correlation.
- omit missing variables or do not include the missing year into the cleaned dataset ##need to double check

The most plausible remaining concern is that time-varying country-specific factors are still correlated with both enrolment and fertility. If those factors are not captured by the fixed effects, the coefficient may still be biased.

## 8. Supplementary Files and Robustness Checks

Supplementary files:
- `../Data.clean/panel_fixed_effects_data.csv`: merged country-year panel dataset used for estimation.
- `../../Scripts/08_fixed_effects_analysis.py`: script that constructs the panel and estimates the two-way fixed effects regression.
- `../../Outputs/tables/fixed_effects_results.txt`: saved regression output for reproducibility.

Suggested robustness checks:
- estimate the same model without year fixed effects to see whether the coefficient changes when common shocks are excluded,
- estimate the model using only Malawi data as a within-country time-series check,
- add additional time-varying controls if available, such as GDP per capita, female labor force participation, or health expenditure,
- compare the main sample with alternative country sets to assess sensitivity to sample selection.